In [2]:
import json
import re
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableMap, RunnablePassthrough
from datasets import Dataset

In [4]:
# 1) 환경 변수 로드
load_dotenv()

True

In [5]:
# 2) LLM, 파서 준비
llm = ChatOpenAI(model_name="gpt-4.1", openai_api_key=OPENAI_API_KEY)
parser = StrOutputParser()

In [ ]:
# 3가지 충북지역 추천
def region_recommend(query : str):
    prompt = f"""
아래 "질문"에서 유저의 유저정보, 중요도를 파싱해서
대한민국 충청북도 지방 중 이 유저가 정착하면 좋을 지역 3곳을 추천해줘..
각 지역은 다음 기준에 따라 간단한 근거도 함께 설명해줘.  

**한 유저의 프로필과 지역 선호 정보 예시**:
[유저 정보]
- 나이: 28세
- 성별: 여성
- 관심 직무: 식품 관련 창업
- 월 생활비 가능 금액: 60만원

[중요도]
- 집 값: 매우 중요
- 교통: 중요
- 의료시설: 보통
- 교육 인프라: 중요
- 인구 밀도: 낮은 것을 선호
- 자연환경: 매우 중요


**이유 요약 예시**:
- 예산 부담이 적음
- 관련 직무 인프라와 가능성 있음
- 선호한 인프라 조건과의 일치도

응답 형식:
[
"region": "지역명",
"reason": "이유 요약"
]

질문: {query}
반드시 위 응답 형식을 따라.

"""
    response = llm.invoke(prompt)
    return response

In [ ]:
query = " 나이: 28세 / 성별: 여성 / 관심 직무: 간호 / 월 생활비 가능 금액: 90만원 / 집 값: 매우 중요 / 교통: 보통 / 의료시설: 중요 /  교육 인프라: 보통 / 인구 밀도: 낮은 것을 선호 / 자연환경: 매우 중요"

response = region_recommend(query)

In [18]:
print(response.content)

[
  {
    "region": "제천시",
    "reason": "집값이 비교적 저렴하고, 대형병원을 포함한 의료 인프라가 잘 갖춰짐. 인구 밀도가 낮고 자연환경(청풍호, 산 등)이 우수해 생활비 안에서 쾌적한 정착이 가능."
  },
  {
    "region": "단양군",
    "reason": "집값이 저렴하고 인구 밀도가 매우 낮아 한적한 생활이 가능함. 최근 의료시설 개선 노력과 깨끗한 자연환경(도담삼봉, 산, 강)으로 자연 친화적이고 여유로운 정착에 적합."
  },
  {
    "region": "옥천군",
    "reason": "주거 비용 부담이 적고 의료기관이 고르게 분포, 교통 접근성도 청주와 가까워 비교적 용이함. 산과 계곡 등 자연환경이 뛰어나 삶의 질 중시 조건에 부합."
  }
]


In [65]:
parsed = json.loads(response.content)
type(parsed)

list

In [93]:
region_list = [item["region"] for item in parsed]
region_list

['제천시', '옥천군', '단양군']

### LLM에서 받은 3가지 지역 / 유저 입력 쿼리에서 뽑아낸 나이, 성별, 직무로 metadata 필터 만들기

- 성별, 직무에 관한 정책은 좀 애매한거 같기도 ?
- 나이, 소득으로 가장 많이 갈리지 않나

In [45]:
# 유저 쿼리 파싱
def parse_user_query(query):
    parsed = {}
    parsed["age"] = int(re.search(r"나이:\s*(\d+)", query).group(1))
    parsed["gender"] = re.search(r"성별:\s*(\w+)", query).group(1)
    parsed["job_interest"] = re.search(r"관심 직무:\s*([\w\s]+)", query).group(1).strip()
    return parsed

In [63]:
user_info = parse_user_query(query)
user_info

{'age': 28, 'gender': '여성', 'job_interest': '간호'}

In [3]:
df = pd.read_csv("../data/youth_policies_all.csv", encoding="utf-8-sig")

In [4]:
df.rename(columns={"sprtTrgtMinAge": "age_min", "sprtTrgtMaxAge" : "age_max"}, inplace=True)

In [5]:
df.tail(2)

,plcyNm,plcyExplnCn,plcySprtCn,aplyYmd,aplyUrlAddr,age_min,age_max,lclsfNm,mclsfNm
3676,청년마음건강지원사업 이용자 모집(김해시),"우울감, 취업 애로 등으로 심리적 어려움을 겪고 있는 전국 청년을 대상으로 전문심리...",□ 지원내용: 3개월 간 총 10회의 전문심리상담서비스 제공\r\nㅇ 사전사후검사:...,20240108 ~ 20240112,https://www.bokjiro.go.kr/,19.0,34.0,"복지문화,복지문화","취약계층 및 금융지원,건강"
3677,2025년 강릉시 청년창업 희망키움 사업,강릉지역 청년 (예비)창업자 모집공고강릉지역 창업 3년 미만의 청년(예비) 창업기업...,"- 창업 사업화 자금 1,000만원~최대 1,600만원 지원 \r\n * (1단계...",20250301 ~ 20250328,https://www.gn.go.kr/www/selectBbsNttView.do?k...,18.0,45.0,일자리,취업


In [90]:
# 정책 필터링 함수
def get_matched_policies(user_info, region_list, policy_df):
    matched = []

    for region in region_list:
        # 해당 지역에 대한 정보라 정책DB에 지역이름이 들어가야 함.
        ## 근데 충북지역으로 일단 진행하는거니 괜찮을 것 같기도
        region_policies = policy_df[policy_df["region"] == region]

        # 나이 필터
        region_policies = region_policies[
            (region_policies["age_min"] <= user_info["age"]) &
            (region_policies["age_max"] >= user_info["age"])
        ]

        # 성별 필터
        ## 성별에 따른 정책이 많이 나뉜다고 명인이가 얘기하긴 했는데 저 DB에서 그걸로 나누긴 힘들 것 같기도
        region_policies = region_policies[
            (region_policies["gender"] == user_info["gender"]) |
            (region_policies["gender"] == "전체")
        ]

        # 직무 키워드 필터
        ## 직무도 사실 모르겠음 어떤 식으로 필터링 걸어서 정책 뿌려줘야 할지
        region_policies = region_policies[
            region_policies["job_keywords"].apply(lambda keywords: any(
                kw in user_info["job_interest"] for kw in keywords))
        ]

        matched.append(region_policies[["policy_name", "summary"]].to_dict(orient="records"))

    return matched

In [101]:
# 예시 DB
policy_df = pd.DataFrame([
    {
        "region": "제천시",
        "policy_name": "청년 간호사 정착 지원",
        "age_min": 20,
        "age_max": 35,
        "gender": "전체",
        "job_keywords": ["간호", "보건", "의료"],
        "summary": "제천 의료기관 취업 간호사 대상 1년간 월 20만원 지원"
    },
    {
        "region": "옥천군",
        "policy_name": "간호직 여성 일자리 사업",
        "age_min": 25,
        "age_max": 34,
        "gender": "여성",
        "job_keywords": ["간호", "의료"],
        "summary": "옥천군 간호직 여성 대상 정규직 취업 연계 프로그램"
    },
    {
        "region": "단양군",
        "policy_name": "청년 보건직 창업 지원",
        "age_min": 24,
        "age_max": 39,
        "gender": "전체",
        "job_keywords": ["보건", "간호", "의료"],
        "summary": "보건 관련 창업 시 최대 1000만원 지원"
    },
    {
        "region": "단양군",
        "policy_name": "간호 몰루 지원",
        "age_min": 19,
        "age_max": 40,
        "gender": "전체",
        "job_keywords": ["간호"],
        "summary": "간호 1년차 정착비 지원"
    }
])

In [103]:
matched_policies = get_matched_policies(user_info, region_list, policy_df)

matched_policies

[[{'policy_name': '청년 간호사 정착 지원',
   'summary': '제천 의료기관 취업 간호사 대상 1년간 월 20만원 지원'}],
 [{'policy_name': '간호직 여성 일자리 사업', 'summary': '옥천군 간호직 여성 대상 정규직 취업 연계 프로그램'}],
 [{'policy_name': '청년 보건직 창업 지원', 'summary': '보건 관련 창업 시 최대 1000만원 지원'},
  {'policy_name': '간호 몰루 지원', 'summary': '간호 1년차 정착비 지원'}]]

In [ ]:
# 결과 출력
## 아니 근데 이건 rag가 아니잖아 생각해보니;
for region, policies in zip(region_list, matched_policies):
    print(f"{region}")
    if policies:
        for p in policies:
            print(f"- {p['policy_name']}: {p['summary']}")
    else:
        print("- 해당 조건에 맞는 정책이 없습니다.")

제천시
- 청년 간호사 정착 지원: 제천 의료기관 취업 간호사 대상 1년간 월 20만원 지원
옥천군
- 간호직 여성 일자리 사업: 옥천군 간호직 여성 대상 정규직 취업 연계 프로그램
단양군
- 청년 보건직 창업 지원: 보건 관련 창업 시 최대 1000만원 지원
- 간호 몰루 지원: 간호 1년차 정착비 지원


- 근데 우리가 data가 저렇게 깔끔한게 아니고 rag 쓸 꺼니까 흠 일단 데이터를 좀 깎긴해야할듯 ? 그래야 임베딩할 때 메타데이터로 뽑아낼꺼 뽑아내고 임베딩을 해야되니까
- 흠 아니 rag로 해야하나 ??